In [8]:
lastest_json = '/allah/stuff/freq/userdir/backtest_results/backtest-result-2024-12-21_11-06-55.json'
import json
with open(lastest_json) as f:
    data = json.load(f)
data['strategy']['MultiTimeframeTEMAAgreement']['trades'][-1]

{'pair': 'COW/USDT:USDT',
 'stake_amount': 0.069888,
 'max_stake_amount': 5.2416,
 'amount': 8.0,
 'open_date': '2024-12-19 15:47:00+00:00',
 'close_date': '2024-12-19 15:47:00+00:00',
 'open_rate': 0.6552,
 'close_rate': 0.6553,
 'fee_open': 0.0005,
 'fee_close': 0.0005,
 'trade_duration': 0,
 'profit_ratio': -0.08649585781901939,
 'profit_abs': -0.006042,
 'exit_reason': 'stop_loss',
 'initial_stop_loss_abs': 0.6553,
 'initial_stop_loss_ratio': -0.02,
 'stop_loss_abs': 0.6553,
 'stop_loss_ratio': -0.02,
 'min_rate': 0.6551,
 'max_rate': 0.658,
 'is_open': False,
 'enter_tag': 'aligned_down',
 'leverage': 75.0,
 'is_short': True,
 'open_timestamp': 1734623220000,
 'close_timestamp': 1734623220000,
 'orders': [{'amount': 8.0,
   'safe_price': 0.6552,
   'ft_order_side': 'sell',
   'order_filled_timestamp': 1734623220000,
   'ft_is_entry': True,
   'ft_order_tag': 'aligned_down'},
  {'amount': 8.0,
   'safe_price': 0.6553,
   'ft_order_side': 'buy',
   'order_filled_timestamp': 17346232

In [ ]:
# read backtest-result-2024-12-14_11-37-27.json
# backtest-result-2024-12-14_11-37-27.meta.json
# backtest-result-2024-12-14_11-37-27_market_change.feather

import json
import pandas as pd
with open('/allah/stuff/freq/userdir/backtest_results/backtest-result-2024-12-14_16-48-36.json', 'r') as f:
    long_backtest_results = json.load(f)
# Load the backtest results
with open('/allah/stuff/freq/userdir/backtest_results/backtest-result-2024-12-14_18-19-02.json', 'r') as f:
    short_backtest_results = json.load(f)


long_trades = long_backtest_results["strategy"]["TEMA50TrailingStopStrategy"]["trades"]
short_trades = short_backtest_results["strategy"]["TEMA50TrailingStopStrategy"]["trades"]

df_trades_long = pd.DataFrame(long_trades)
df_trades_short = pd.DataFrame(short_trades)

In [ ]:
import pandas as pd

# Assuming df_trades_long and df_trades_short are already loaded with your data

# Fixing the typo in column names for clarity
df_trades_long['is_long_profit'] = df_trades_long['profit_abs'] > 0
df_trades_long['long_profitability'] = df_trades_long['profit_abs']

df_trades_short['is_short_profit'] = df_trades_short['profit_abs'] > 0
df_trades_short['short_profitability'] = df_trades_short['profit_abs']

# Merging the DataFrames based on a common column 'open_date' if it's unique or an appropriate key
# Ensure that 'open_date' is a suitable key for merging; if not, adjust accordingly
df_combined = pd.merge(df_trades_long, df_trades_short[['open_date', 'is_short_profit', 'short_profitability']],
                       on='open_date', how='left')

# Selecting only the required columns for the final DataFrame
final_df = df_combined[['open_date', 'is_long_profit', 'long_profitability', 'is_short_profit', 'short_profitability', 'enter_tag']]


In [ ]:
final_df['open_date'] = pd.to_datetime(final_df['open_date'], utc=True)

final_df['target_date'] = final_df['open_date']  - pd.Timedelta(minutes=3)

In [ ]:
final_df

In [ ]:
final_df['target_date']

In [ ]:
print(final_df['open_date'].dtype)

In [ ]:
final_df

In [ ]:
# Filtering the DataFrame for rows where is_long_profit is True and is_short_profit is False
filtered_df = final_df[(final_df['type'] == 'LONG_WIN') & (final_df['enter_tag'] == 'tema50_up')]
# Filtering the DataFrame for rows where is_long_profit is True and is_short_profit is False
filtered_df.head(20)

final_df.type.value_counts()



In [ ]:
# Assuming final_df is loaded with your data
# Calculate the counts for each combination
count_both_false = len(final_df[(final_df['is_long_profit'] == False) & (final_df['is_short_profit'] == False)])
count_both_true = len(final_df[(final_df['is_long_profit'] == True) & (final_df['is_short_profit'] == True)])
count_long_true_short_false = len(final_df[(final_df['is_long_profit'] == True) & (final_df['is_short_profit'] == False)])
count_long_false_short_true = len(final_df[(final_df['is_long_profit'] == False) & (final_df['is_short_profit'] == True)])

# Print out the results
print("Both long and short profit False:", count_both_false)
print("Both long and short profit True:", count_both_true)
print("Long profit True and short profit False:", count_long_true_short_false)
print("Long profit False and short profit True:", count_long_false_short_true)

import numpy as np

# Define conditions
conditions = [
    (final_df['is_long_profit'] == False) & (final_df['is_short_profit'] == False), # Both false
    (final_df['is_long_profit'] == True) & (final_df['is_short_profit'] == True),   # Both true
    (final_df['is_long_profit'] == True) & (final_df['is_short_profit'] == False),  # Long win only
    (final_df['is_long_profit'] == False) & (final_df['is_short_profit'] == True)   # Short win only
]

# Define choices corresponding to the conditions
choices = [
    'LOSE',     # Both lose
    'WIN',      # Both win
    'LONG_WIN', # Only long win
    'SHORT_WIN' # Only short win
]

# Create a new column 'type' with values based on conditions
final_df['type'] = np.select(conditions, choices, default='undefined')

# Print or view the DataFrame to confirm the new column is added correctly
final_df.to_parquet("/allah/data/parquet/final_df.parquet")


In [ ]:
df_trades_long['long_profitability']

In [ ]:
df_trades_short[['enter_tag', 'is_short', 'profit_abs', 'open_date']]